## Ch10-02 -- A judgment ledger, over three real records this tutorial has already built

This notebook introduces a judgment ledger over three real `ReviewRecord`s already built earlier in this tutorial; after running it you can see what each one claims, which of Hawkins' three judgment kinds it represents, and what its own `residual_uncertainties` says is not yet resolved.

This tutorial has no central registry of every `ReviewRecord` it has ever built, the same finding Chapter 9 made when it searched for one: each chapter's notebook constructs its own records as local Python objects, per the construction-zone pattern (`toaster-review-protocol`), with nothing shared beyond the file each one lives in. So this notebook does not scan "every record the tutorial has ever produced"; instead it reconstructs three real records verbatim, field for field: `AS-C06` and `AS-C08`, already reconstructed once by Chapter 9 and re-verified here against their real originals rather than assumed correct just because Chapter 9 said so, plus one more, `AI-C06` (Chapter 6's own stopping judgment over the level-2 heat-generation branch, an `asserted_inference`, alongside the two `asserted_solution` records Chapter 9 already carried forward). What follows demonstrates the mechanics of a judgment ledger on a small, representative sample, not an exhaustive audit of every judgment record this tutorial has ever produced.

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch10-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"


Before reconstructing the three records, the negative control fitting this notebook's own new record, `AI-C06`: Hawkins SS3.1 requires an `asserted_inference` to name at least one premise, since a bare assertion with nothing supporting it is not an inference at all. `validate_record()` already enforces this.

In [2]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

incomplete_inference = ReviewRecord(
    identifier="AI-BAD",
    kind="asserted_inference",
    claim="The judgment ledger is complete.",
    subject_ref="ToasterDemo::HeatingAssembly::heatGen",
    model_ref="ToasterDemo",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="Every real ReviewRecord this tutorial has built is included.",
    premises=[],   # intentionally empty
    rationale="Chapter 10 built a ledger.",
    counterevidence="No central registry exists to check completeness against.",
    record_kind="worked_example",
)
errors = validate_record(incomplete_inference, model=model)
assert errors == ["asserted_inference requires at least one premise (Hawkins \u00a73.1)"], errors
print(f"Negative control ok: errors={errors}")


Negative control ok: errors=['asserted_inference requires at least one premise (Hawkins §3.1)']


`AS-C06`, rebuilt below **verbatim** from Chapter 6 notebook 02's own field strings, the same reconstruction Chapter 9 already carried forward: a mechanism-selection judgment, argued from a domain premise about how a resistive element and a combustion burner each respond to a discrete timing signal. Its `content_hash` is computed against `models/ch06-cumulative.sysml`, the real model it was actually written against.

In [3]:
ch06_source = Path("../../models/ch06-cumulative.sysml").read_text()

as_c06 = ReviewRecord(
    identifier="AS-C06",
    kind="asserted_solution",
    claim=(
        "ResistanceCoil, an electrically switched resistive element that converts "
        "energy to heat by Joule heating, is selected over a combustion-based "
        "alternative (a gas burner, the tongs-and-blowtorch alternative this "
        "tutorial already contrasts) as the mechanism HeatGenerator commits to."
    ),
    subject_ref="ToasterDemo::ResistanceCoil",
    model_ref="ToasterDemo::ResistanceCoil",
    content_hash=hash_content(ch06_source),
    scope="ToasterDemo::HeatGenerator and its realizations",
    criteria=(
        "The chosen mechanism must pair with the discrete timing control "
        "ControlSystem's durationOut already provides, and must expose a rating "
        "heatGenerationReq's power threshold can be checked against once a "
        "concrete part exists."
    ),
    premises=[
        "ControlSystem declares durationOut : DurationPort (Chapter 5), a "
        "discrete duration signal, confirmed by model.find() above.",
        "Domain premise, not derived from the model: a resistive element "
        "responds to being switched on and off directly, while a combustion "
        "source needs separate ignition and fuel-metering machinery to do "
        "the same. Neither HeatGenerator nor any of its realizations is yet "
        "connected to ControlSystem's port in this model; this premise is "
        "about the physical mechanisms themselves, not about what the model "
        "currently wires together.",
    ],
    assumption_refs=[
        "HeatGenerator::energyIn and GenerateHeat carry no energy-form commitment "
        "(notebook 01): this record is what actually commits to an electrical "
        "form, not a fact already built into the port or the function."
    ],
    evidence_refs=[
        "model.find('ToasterDemo::ControlSystem::durationOut') resolves to a "
        "real portUsage, confirmed above."
    ],
    rationale=(
        "An electrically resistive element responds to being switched on "
        "and off directly, the same shape as duration's discrete timing "
        "signal, while a combustion-based burner needs separate ignition "
        "and fuel-metering machinery to respond the same way (the domain "
        "premise above). That is a claim about how the two mechanisms "
        "work, not something this model currently shows: no realization "
        "of HeatGenerator is yet connected to ControlSystem's "
        "durationOut port, so this selection is a reasoned engineering "
        "preference argued from mechanism, not a claim that the model "
        "already connects one mechanism and not the other."
    ),
    counterevidence=(
        "This does not rule out a combustion design: a burner controlled by its "
        "own timed valve could equally use a duration-like signal, which is "
        "exactly why the argument above rests on a domain premise about how "
        "the two mechanisms work, not on anything the model itself already "
        "builds or connects. Joule heating's own relation (power proportional "
        "to resistance and the square of current) is still not modeled, so "
        "efficiency and response-time comparisons remain out of reach "
        "either way."
    ),
    residual_uncertainties=(
        "Once a supply and a control policy are modeled together, this "
        "selection could be revisited against a real trade study rather than "
        "a domain premise alone."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(as_c06, model=model)
print(f"AS-C06 validation errors: {errors}")


AS-C06 validation errors: []


`AS-C08`, rebuilt the same way, verbatim, from Chapter 8 notebook 02: the record grounded in that chapter's real Z3 proof of `deliveredEnergyBoundedBySupply`, the same one notebook 01 of this chapter found disconnected from any requirement usage and then tied, by subsetting, to `energyConservationReq`. Its `content_hash` is computed against `models/ch08-cumulative.sysml`, the model it was actually written against.

In [4]:
ch08_source = Path("../../models/ch08-cumulative.sysml").read_text()

as_c08 = ReviewRecord(
    identifier="AS-C08",
    kind="asserted_solution",
    claim=(
        "deliveredEnergyBoundedBySupply, a hand-restated real-arithmetic lemma of the same "
        "shape as HeatGenerator's efficiencyBounded constraint and deliveredEnergy's own "
        "definition, holds for every value of efficiency in [0,1] and every non-negative "
        "power and duration a companion restatement admits."
    ),
    subject_ref="ToasterDemo::deliveredEnergyBoundedBySupply",
    model_ref="ToasterDemo::deliveredEnergyBoundedBySupply",
    content_hash=hash_content(ch08_source),
    scope=(
        "The lemma governs the companion restatement's own heatGenCheck.efficiency, "
        "heatGenCheck.power and heatGenCheckDuration features; it is not a solver-checked "
        "reference to HeatGenerator's own efficiencyBounded or deliveredEnergy (DEFERRED.md "
        "D-030, D-031), only a hand-restated copy of the same shape."
    ),
    criteria=(
        "verify_holds() reports a single satisfied verdict for deliveredEnergyBoundedBySupply, "
        "with the reason text naming z3 (not propagation alone), proved for all values of the "
        "unbound heatGenCheck.efficiency, heatGenCheck.power and heatGenCheckDuration features."
    ),
    premises=[],
    assumption_refs=[],
    evidence_refs=[
        "verify_holds: deliveredEnergyBoundedBySupply satisfied "
        "(z3: holds for all values of unbound features)"
    ],
    rationale=(
        "verify_holds() calls sysml-toolkit's real verify --solve command, which runs Z3 "
        "over the unbound features of a companion restatement of this lemma (see the "
        "narration above for why a companion file is used) and reports "
        "deliveredEnergyBoundedBySupply as satisfied: proved for all values the restatement "
        "admits, not read back from one entered value. This is a materially different kind of "
        "evidence from an evaluate-only verdict: verify_satisfaction() could only ever check "
        "a relation at whichever single power, duration and efficiency a candidate happens to "
        "carry. It is also, deliberately, a narrower claim than 'this proves HeatGenerator's "
        "own conservation property': see counterevidence."
    ),
    counterevidence=(
        "This proof is NOT a solver-checked reference to HeatGenerator's own "
        "efficiencyBounded constraint or deliveredEnergy calc: this toolchain's Z3 backend "
        "does not compose two separately declared assert constraints, whether sibling or "
        "inherited (DEFERRED.md D-030), and cannot reason through a chained calc invocation "
        "like heatGenCheck.deliveredEnergy(...) (D-031). Confirmed directly: loosening "
        "efficiencyBounded's own literal bound to <= 1.5, or doubling deliveredEnergy's own "
        "definition by a factor of 2.0, in the real committed model changes neither the "
        "original elements' own verdicts nor this lemma's verdict at all, because the lemma "
        "restates its own copy of both rather than referencing either. The companion file "
        "used by verify_holds also restates the construct rather than checking the "
        "committed model directly, because toaster.modelcheck's own text parser does not "
        "yet handle the extra annotation the CLI prints for assert satisfy declarations "
        "(DEFERRED.md D-029)."
    ),
    residual_uncertainties=(
        "Whether efficiency, power and duration ever take values outside the bound in a "
        "real candidate is not addressed by this proof; it establishes only that the "
        "restated lemma respects conservation wherever its own bound is honored. If "
        "HeatGenerator's own efficiencyBounded or deliveredEnergy is ever edited, this "
        "record's content_hash (computed from the whole model file) does go stale, which "
        "forces a re-review, but nothing automatically re-checks that the restated copy "
        "still matches the edited original; that check would be manual. No physical heat "
        "generator has been checked against this property; HeatGenerator remains an "
        "abstract carrier with no concrete realization of its own."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

errors = validate_record(as_c08, model=model)
print(f"AS-C08 validation errors: {errors}")


AS-C08 validation errors: []


`AI-C06`, the third record, is genuinely different in kind: an `asserted_inference` about the level-2 heat-generation branch's own stopping judgment, rebuilt verbatim from Chapter 6 notebook 03. Its `premises` field names four earlier judgment identifiers directly (`AC-C06`, `AS-C06`, `AS-C03`, `AI-C04`), the clearest example in this tutorial of Hawkins' own "child claims supporting a parent" idea. Its `evidence_refs` cites the same real analysis results Chapter 6 notebook 03 gathered against that chapter's own model, recomputed below against a freshly loaded `models/ch06-cumulative.sysml` rather than restated by hand. Its `content_hash` is also computed against that same file, since both `AI-C06` and `AS-C06` above were written against Chapter 6's own model.

In [5]:
from toaster.query import find_allocations, perform_relationships

ch06_model = conn.load_from_content(ch06_source, strict=False)
assert ch06_model.ok

performs = perform_relationships(ch06_model)
allocations = find_allocations(ch06_model)
rated_holds = ch06_model.eval("ToasterDemo::heatGenerationReq(ToasterDemo::rated)")
weak_holds = ch06_model.eval("ToasterDemo::heatGenerationReq(ToasterDemo::weak)")
print(f"Perform relationships: {performs}")
print(f"Allocations: {allocations}")
print(f"heatGenerationReq(rated) = {rated_holds}, heatGenerationReq(weak) = {weak_holds}")


Perform relationships: [{'performer': 'ToasterDemo::ToastingSystem', 'action': 'ToasterDemo::ToastBread'}, {'performer': 'ToasterDemo::HeatingSystem', 'action': 'ToasterDemo::ApplyHeat'}, {'performer': 'ToasterDemo::HeatGenerator', 'action': 'ToasterDemo::GenerateHeat'}]
Allocations: [{'id': 'ToasterDemo::Toaster::heatAllocation', 'type': 'AllocationUsage', 'ends': [['ToasterDemo::ToastingSystem::toastBread', 'ToasterDemo::ToastBread::applyHeat'], ['ToasterDemo::Toaster::heating']]}, {'id': 'ToasterDemo::HeatingAssembly::heatGenAllocation', 'type': 'AllocationUsage', 'ends': [['ToasterDemo::HeatingSystem::applyHeat', 'ToasterDemo::ApplyHeat::generateHeat'], ['ToasterDemo::HeatingAssembly::heatGen']]}]
heatGenerationReq(rated) = True, heatGenerationReq(weak) = False


`HeatGenerator` performs `GenerateHeat`, `heatGenAllocation` points from `applyHeat.generateHeat` to `heatGen`, declared inside `HeatingAssembly` where both resolve, and the requirement evaluates True on `rated` and False on `weak`, the same three real results Chapter 6 notebook 03 gathered. The record below assembles from these results directly, the same construction-zone pattern used for `AS-C06` and `AS-C08` above.

In [6]:
ai_c06 = ReviewRecord(
    identifier="AI-C06",
    kind="asserted_inference",
    claim=(
        "GenerateHeat is a real, specified behavior: HeatGenerator performs it "
        "and is allocated the nested step, not merely declared syntax. energyIn "
        "is a declared, typed connection point on HeatGenerator, not yet wired "
        "to a producer. heatGenerationReq evaluates against two real "
        "candidates, rated (True) and weak (False), a genuine satisfaction "
        "check on an underived threshold, not an unevaluated assertion."
    ),
    subject_ref="ToasterDemo::HeatingAssembly::heatGen",
    model_ref="ToasterDemo::HeatingAssembly::heatGen",
    content_hash=hash_content(ch06_source),
    scope="ToasterDemo::HeatingAssembly::heatGen and its realizations",
    criteria=(
        "Per the recursion's own stopping rule, a leaf performs its specified "
        "behavior, connects through its specified interfaces, and has "
        "verification evidence. At this level: GenerateHeat is a specified "
        "behavior, allocated to HeatGenerator (met). energyIn is a declared, "
        "typed connection point, not yet connected to any producer "
        "(partially met, not complete). heatGenerationReq evaluates on two "
        "real candidates against a threshold that is itself not yet derived "
        "(partial evidence, not full verification)."
    ),
    premises=["AC-C06", "AS-C06", "AS-C03", "AI-C04"],
    assumption_refs=[
        "GenerateHeat's own energy input is bound to ApplyHeat::energy, "
        "printed as part of APPLY_HEAT_INCREMENT and loaded successfully in "
        "notebook 01; this does not by itself mean energyIn is wired to any "
        "producer."
    ],
    evidence_refs=[
        f"perform_relationships(model): {performs}",
        f"find_allocations(model): {allocations}",
        f"heatGenerationReq(rated) = {rated_holds}, heatGenerationReq(weak) = "
        f"{weak_holds}, evaluated directly against the loaded model above.",
    ],
    rationale=(
        "Performs: real, not merely declared. HeatGenerator performing "
        "GenerateHeat and heatGenAllocation both appear directly in "
        "perform_relationships and find_allocations above, not just in the "
        "source text, the same performer-and-allocation split Chapter 5 "
        "established one level up. Connects: energyIn is declared and typed, "
        "the interface point the stopping rule names, but it is not wired to "
        "any producer, so this condition is only partially met. Verified: "
        "heatGenerationReq is genuinely evaluated, not left as an unevaluated "
        "assertion, on two real candidates, one passing and one deliberately "
        "failing for a reason about the design; but its own threshold is not "
        "yet derived from any stated measure of effectiveness (AC-C06), so "
        "this is partial verification evidence, not a completed check."
    ),
    counterevidence=(
        "energyIn has no producer wired to it: no supply or wire component "
        "exists in this model, so the interface point is declared, not yet "
        "connected end to end, the same partial state Chapter 5 left "
        "ApplyHeat::duration in before that chapter built its own port "
        "connection. HeatingAssembly is not yet composed into any Toaster "
        "candidate: Toaster::heating is still typed by the abstract "
        "HeatingSystem, so no full toaster candidate contains a resistance "
        "coil yet. The 600 W threshold is not derived from any stated "
        "measure of effectiveness (AC-C06); it is a free-standing "
        "engineering figure. This branch addresses only GenerateHeat's own "
        "energy-to-heat conversion: ApplyHeat's other flows (bread, "
        "duration, toast, delivered, loss) are not decomposed or accounted "
        "for at this level. This is a deliberate one-branch worked example "
        "of one of ApplyHeat's own flows, the same kind of honestly scoped "
        "choice Chapter 4 made for ApplyHeat itself out of Douglas's roughly "
        "fifteen functions, not a claim that Chapter 6 finishes ApplyHeat's "
        "full decomposition."
    ),
    residual_uncertainties=(
        "Whether GenerateHeat needs further decomposition of its own, "
        "whether HeatingAssembly is ever composed into a real Toaster "
        "candidate, and whether ApplyHeat's other flows (duration, bread, "
        "toast) get their own level-2 branches, are open for whichever "
        "chapter takes them up."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(ai_c06, model=model)
print(f"AI-C06 validation errors: {errors}")


AI-C06 validation errors: []


All three validate cleanly. The ledger below reports each record's kind (Hawkins' three sites: `asserted_context`, `asserted_inference`, `asserted_solution`), its `disposition` and `record_kind` (SA-7: always `pending`, always `worked_example`, never the forbidden alternatives), and its `engineering_conclusion`, next to what its own `residual_uncertainties` says is NOT yet resolved.

In [7]:
ledger = {"AS-C06": as_c06, "AS-C08": as_c08, "AI-C06": ai_c06}

for identifier, record in ledger.items():
    print(f"{identifier}:")
    print(f"  kind: {record.kind}")
    print(f"  disposition: {record.disposition}")
    print(f"  record_kind: {record.record_kind}")
    print(f"  engineering_conclusion: {record.engineering_conclusion}")
    print(f"  residual_uncertainties: {record.residual_uncertainties}")
    print()

assert all(r.disposition == "pending" for r in ledger.values())
assert all(r.record_kind == "worked_example" for r in ledger.values())
conn.close()


AS-C06:
  kind: asserted_solution
  disposition: pending
  record_kind: worked_example
  engineering_conclusion: undetermined
  residual_uncertainties: Once a supply and a control policy are modeled together, this selection could be revisited against a real trade study rather than a domain premise alone.

AS-C08:
  kind: asserted_solution
  disposition: pending
  record_kind: worked_example
  engineering_conclusion: supported
  residual_uncertainties: Whether efficiency, power and duration ever take values outside the bound in a real candidate is not addressed by this proof; it establishes only that the restated lemma respects conservation wherever its own bound is honored. If HeatGenerator's own efficiencyBounded or deliveredEnergy is ever edited, this record's content_hash (computed from the whole model file) does go stale, which forces a re-review, but nothing automatically re-checks that the restated copy still matches the edited original; that check would be manual. No physical he

Three real records, three different points of confidence. `AS-C06` stays `undetermined`: its own residual admits the mechanism selection could be revisited against a real trade study once a supply and control policy are modeled together, not settled by the domain premise alone. `AS-C08` is `supported`, but narrowly: its own residual is explicit that the restated lemma is not automatically re-checked against the real elements it mirrors if either is edited, and that no physical heat generator has ever been checked against it. `AI-C06` stays `undetermined` too: its own residual leaves open whether `GenerateHeat` needs further decomposition, whether `HeatingAssembly` is ever composed into a real `Toaster` candidate, and whether `ApplyHeat`'s other flows get their own branches. Two records out of three stay `undetermined`, and the one `supported` record is supported only for a claim already narrowed to a hand-restated copy, not the real elements it mirrors. That is what a ledger is for: not a tally of how many records exist, but a reading of how much of that count is actually load-bearing.

Building three real judgment records side by side, reading their own kind, disposition and residual uncertainty rather than just their count, is what turns "this tutorial has produced some ReviewRecords" into a ledger an accountable engineer could actually use: what follows next, in [notebook 03](03-engineering-signoff.ipynb), synthesizes this ledger together with notebook 01's own traceability graph.

Try the chapter exercise in `exercises/ch10/exercise.ipynb`: it asks you to build this same traceability graph and judgment ledger over your own coffee-maker model, then synthesize both into one honestly-scoped sign-off record.